<a href="https://colab.research.google.com/github/szuhunghsiao/multi-agent-research-assistant/blob/main/multi_agent_research_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q langgraph langchain-core langchain-google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.7/571.7 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.0/260.0 kB 20.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.57.1 which is incompatible.


In [ ]:
import langgraph, langchain_core
from importlib.metadata import version
print("langgraph:", version("langgraph"))
print("langchain-core:", langchain_core.__version__)
# print("langchain-google-genai:", version("langchain-google-genai"))

langgraph: 1.2.11
langchain-core: 1.6.0


In [ ]:
from google.colab import userdata
import os
key_names = ["Researcher", "Writer", "Critic"]
for name in key_names:
  val = userdata.get(name)
  os.environ[name] = val
  print(f"{name}: {'OK, len=' + str(len(val)) if val else 'MISSING'}")

Researcher: OK, len=53
Writer: OK, len=53
Critic: OK, len=53


In [ ]:
from typing import TypedDict, List
class ResearchState(TypedDict):
  topic: str            # User input topic
  research_notes: str   # Researcher generate info clearn up
  draft: str            # Writer current draft
  critique: str         # Critic latest judge opinion
  revision_count: int   # Current correctness times, for controling loop ending
  approved: bool        # Critic passing or not

print("State Def result complete")

State Def result complete


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
researcher_llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    google_api_key=os.environ["Researcher"],
)

def extract_text(response) -> str:
  """
    Dealing with Gemini return format (new version could be list of dict, or number)
  """
  content = response.content
  if isinstance(content, str):
    return content
  if isinstance(content, list):
    return "".join(
        block.get("text", "")
        for block in content
        if isinstance(block,dict)
    )
  return str(content)

def researcher_node(state: ResearchState) -> dict:
  prompt = (
      f"You are a research assistant, that focuing on topic '{state['topic']}', "
      f"List out 5 to 8 key facts, background, or anything need to concider with clear, high volume info"
      "Don't include unnecessary details."
  )
  response = researcher_llm.invoke(prompt)
  return {"research_notes": extract_text(response)}

# Quick test
test_state = {"topic": "Current Taiwan electrical motorbike situation",
              "research_notes": "",
              "draft": "",
              "critique": "",
              "revision_count": 0,
              "approved": False}
result = researcher_node(test_state)
print(type(result["research_notes"]))
print(result["research_notes"][:200])


<class 'str'>
Here are 7 high-volume, key facts and background considerations regarding the current Taiwan electrical motorbike (e-motorbike) situation:

1. **Market Dominance of Gogoro (The "Apple of E-Motorbikes"


In [ ]:
writer_llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    google_api_key=os.environ["Writer"],
)

def writer_node(state: ResearchState) -> dict:
  if state.get("critique"):
    # Modification Loop: regarding to Critic to modify previous draft
    prompt = (
        f"This is the draft that you wrote regarding to the topic '{state['topic']}'"
        f"Critiquer gave the opinion as follow: \n{state['critique']}\n\n"
        f"Please modify the draft, and output the corrected result."
        "Not only what you changed, but the complete draft"
    )
  else:
    # First Loop: draft according to the topic
    prompt = (
        f"You are a professional writer. Regarding to the topic: '{state['topic']}'"
        f"Write a clear structured, 300-500 words first draft: \n\n{state['research_notes']}"
    )
  response = writer_llm.invoke(prompt)
  return {
      "draft": extract_text(response),
      "revision_count": state["revision_count"] + (1 if state.get("critique") else 0),
  }

# Quick test follow the previous research_notes
test_state["research_notes"] = result["research_notes"]
draft_result = writer_node(test_state)
print(draft_result["draft"][:300])
print("revision_count:", draft_result["revision_count"])

# Riding the Electric Wave: The Evolution of Taiwan’s E-Motorbike Ecosystem

Taiwan boasts the highest motorcycle density in the world. Today, this bustling two-wheel landscape is undergoing a profound green transformation, driven by ambitious government mandates, pioneering infrastructure, and fier
revision_count: 0


In [ ]:
print("Checking connection")

Checking connection


In [ ]:
critic_llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    google_api_key=os.environ["Critic"],
)

def critic_node(state: ResearchState) -> dict:
    prompt = (
        f"You are a strict checker, check the report draft regarding to the topic:'{state['topic']}':\n\n"
        f"{state['draft']}\n\n"
        f"Check if content is accurate, clear, or if there's any obvious missing point or logic issue.\n"
        f"Please use the following format to reply (first line must be APPROVED or REVISE, not other word in the first line)\n"
        f"APPROVED or REVISE\n"
        f"The following should start the opinion (if it is APPROVED, brifly explain why)"
    )
    response = critic_llm.invoke(prompt)
    text = extract_text(response).strip()

    lines = text.split("\n", 1)
    verdict = lines[0].strip().upper()
    feedback = lines[1].strip() if len(lines) > 1 else ""

    return {
        "approved": verdict.startswith("APPROVED"),
        "critique": feedback,
    }

# 測試：用步驟 5 的 draft 接續測試
test_state["draft"] = draft_result["draft"]
critique_result = critic_node(test_state)
print("approved:", critique_result["approved"])
print("critique:", critique_result["critique"][:300])

approved: False
critique: While your draft is highly engaging, well-structured, and captures the core elements of Taiwan’s e-motorbike landscape, it contains a few factual oversimplifications and misses critical current developments that define the reality of the market in 2023/2024. 

Here is the detailed critique and areas


In [ ]:
from langgraph.graph import StateGraph, START, END

MAX_REVISIONS = 3  # 先隨便設一個上限，下一步再正式處理迴圈終止條件

writer_llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    google_api_key=os.environ["Writer"],
)
critic_llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    google_api_key=os.environ["Critic"],
)

def route_after_critic(state: ResearchState) -> str:
    if state["approved"] or state["revision_count"] >= MAX_REVISIONS:
        return "end"
    return "revise"

graph_builder = StateGraph(ResearchState)
graph_builder.add_node("researcher", researcher_node)
graph_builder.add_node("writer", writer_node)
graph_builder.add_node("critic", critic_node)

graph_builder.add_edge(START, "researcher")
graph_builder.add_edge("researcher", "writer")
graph_builder.add_edge("writer", "critic")
graph_builder.add_conditional_edges(
    "critic",
    route_after_critic,
    {"end": END, "revise": "writer"},
)

graph = graph_builder.compile()
print("Graph Complete Edit")

Graph Complete Edit


In [ ]:
initial_state = {
    "topic": "2026 How AI affect the employment situation",
    "research_notes": "",
    "draft": "",
    "critique": "",
    "revision_count": 0,
    "approved": False,
}

final_state = graph.invoke(initial_state)

print("=== Result ===")
print("Approved:", final_state["approved"])
print("Fix round(s):", final_state["revision_count"])
print("\n=== Final Draft ===")
print(final_state["draft"])
print("\n=== Final Critic Opinion ===")
print(final_state["critique"])

=== Result ===
Approved: True
Fix round(s): 0

=== Final Draft ===
**Navigating the 2026 AI Employment Landscape: Evolution, Disruption, and Adaptation**

As we navigate through 2026, the integration of artificial intelligence into the global economy has moved past speculative hype and into a period of profound structural transformation. Rather than delivering a sudden, apocalyptic wave of total job loss, AI is fundamentally rewriting the mechanics of how we work. Understanding this shift requires looking past the headlines and examining the data-driven realities shaping the modern workforce.

First, the nature of work is experiencing massive task displacement. According to projections from the World Economic Forum and Goldman Sachs, generative AI is automating or significantly augmenting 25% to 30% of all global work tasks. Notably, this disruption targets cognitive and administrative responsibilities far more than manual, blue-collar labor. Knowledge workers—including mid-level analy

In [ ]:
initial_state = {
    "topic": "AI Agent in Enterprise, how employee prevent to be laid off",
    "research_notes": "",
    "draft": "",
    "critique": "",
    "revision_count": 0,
    "approved": False,
}

for event in graph.stream(initial_state):
    for node_name, node_output in event.items():
        print(f"--- State: {node_name} ---")
        if "approved" in node_output:
            print(f"  approved={node_output['approved']}, revision_count={node_output.get('revision_count', '-')}")
        elif "draft" in node_output:
            print(f"  draft (First 80 char): {node_output['draft'][:80]}...")
        elif "research_notes" in node_output:
            print(f"  research_notes (First 80 char): {node_output['research_notes'][:80]}...")
        print()

--- State: researcher ---
  research_notes (First 80 char): Here are 6 key facts, background contexts, and critical considerations regarding...

--- State: writer ---
  draft (First 80 char): **Navigating the AI Shift: How Enterprise Employees Can Future-Proof Their Caree...

--- State: critic ---
  approved=False, revision_count=-

--- State: writer ---
  draft (First 80 char): Here is the complete, revised draft. I have integrated the critiquer's feedback ...

--- State: critic ---
  approved=False, revision_count=-

--- State: writer ---
  draft (First 80 char): Here is the complete, revised draft. It directly addresses all three critiques b...

--- State: critic ---
  approved=True, revision_count=-



In [ ]:
# writer_llm = ChatGoogleGenerativeAI(
#     model="gemini-3.5-flash-lite",
#     google_api_key=os.environ["Writer"],
# )
# critic_llm = ChatGoogleGenerativeAI(
#     model="gemini-3.5-flash-lite",
#     google_api_key=os.environ["Critic"],
# )

def run_research_pipeline(topic: str) -> dict:
    initial_state = {
        "topic": topic,
        "research_notes": "",
        "draft": "",
        "critique": "",
        "revision_count": 0,
        "approved": False,
    }
    final_state = graph.invoke(initial_state)
    return {
        "topic": topic,
        "final_draft": final_state["draft"],
        "approved": final_state["approved"],
        "revision_count": final_state["revision_count"],
        "final_critique": final_state["critique"],
    }

# 測試一次
output = run_research_pipeline("How junior SWE can get hired after laid off in 2026")
print("approved:", output["approved"])
print("revision_count:", output["revision_count"])
print(output["final_draft"][:200])

approved: True
revision_count: 1
Here is the complete, revised draft incorporating all of the critiquer’s feedback. 

Specifically, the following updates were made:
1. **Context/Chronology:** Framed 2026 as the natural evolution of t


In [21]:
!pip install -q gradio

import gradio as gr

def gradio_handler(topic):
    if not topic.strip():
        return "Please enter the research topic", "", ""
    output = run_research_pipeline(topic)
    status = "✅ Approved" if output["approved"] else "⚠️ Meet the max revise rounds, Not Approved"
    meta = f"{status}| revision count:{output['revision_count']}"
    return meta, output["final_draft"], output["final_critique"]

demo = gr.Interface(
    fn=gradio_handler,
    inputs=gr.Textbox(label="Research Topic", placeholder="ex: Battery factory in 2026"),
    outputs=[
        gr.Textbox(label="Status"),
        gr.Markdown(label="Final Report"),
        gr.Textbox(label="Critic Final Opinion"),
    ],
    title="Multi-Agent Research Assistants",
    description="Researcher → Writer → Critic Cooperate Research report (Gemini 3.5 Flash-Lite)",
)

demo.launch(debug=True)

ERROR: Operation cancelled by user
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://8e7fac803939c29785.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://8e7fac803939c29785.gradio.live
